# Project 2: Text Similarity & Duplicate Detection (G05)

## Notebook 1 (Approach 1): TF‑IDF + Logistic Regression (Baseline)

**Goal**: Given a pair of questions, predict whether they are duplicates.

**Why this baseline?**
- TF‑IDF is a strong classic representation for text.
- Logistic Regression is fast, easy to interpret, and provides a reliable first benchmark.

**Pipeline overview**
1. Load dataset and validate required columns.
2. Clean/preprocess text (lowercasing, punctuation removal, stopword removal).
3. Convert text to TF‑IDF vectors.
4. Build a pair feature (absolute difference between the two TF‑IDF vectors).
5. Train Logistic Regression.
6. Evaluate with accuracy + imbalance-aware metrics (ROC‑AUC, PR‑AUC) and a majority baseline.

In [1]:
import os
import re

import numpy as np
import pandas as pd

import nltk
from nltk.corpus import stopwords

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split

In [5]:
print("1. Loading dataset from local CSV file...")

data_path = "/content/sample_data/questions.csv"
if not os.path.exists(data_path):
    raise FileNotFoundError(
        f"Could not find {data_path!r} in the current folder. "
        "Put your dataset CSV next to this notebook or change `data_path`."
    )

df = pd.read_csv(data_path)

#Keep only the columns we actually need for the model
required_cols = ["question1", "question2", "is_duplicate"]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns in CSV: {missing}. Found: {list(df.columns)[:20]}...")

df = df[required_cols].dropna()
print(f"-> Dataset loaded successfully with {df.shape[0]} rows!\n")

1. Loading dataset from local CSV file...
-> Dataset loaded successfully with 49847 rows!



## Step 1.1 — Dataset overview

Before modeling, we quickly profile the dataset to understand:

- **Size and columns** available.
- **Class balance** of `is_duplicate` (how many duplicate vs non-duplicate pairs).
- **Typical question lengths** in tokens.

This helps justify our evaluation choices (e.g., using PR‑AUC) and shows that the train/test split is meaningful.

In [6]:
print("1.1 Dataset profiling...")

print("Shape:", df.shape)
print("Columns:", list(df.columns))

subset = df[["question1", "question2", "is_duplicate"]].copy()
print("\nNull counts:\n", subset.isna().sum())
subset = subset.dropna()
print("After dropna shape:", subset.shape)

#Label distribution
label_counts = subset["is_duplicate"].value_counts().sort_index()
label_props = (label_counts / label_counts.sum()).round(4)
print("\nLabel counts:\n", label_counts)
print("\nLabel proportions:\n", label_props)

#Question length summaries
for col in ["question1", "question2"]:
    lengths = subset[col].astype(str).str.split().str.len()
    print(f"\n{col} token length summary:")
    print(lengths.describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]).round(2))

#Exact-match rate
identical = (
    subset["question1"].astype(str).str.strip().str.lower()
    == subset["question2"].astype(str).str.strip().str.lower()
)
print("\nExact string-match rate:", round(float(identical.mean()), 4))

1.1 Dataset profiling...
Shape: (49847, 3)
Columns: ['question1', 'question2', 'is_duplicate']

Null counts:
 question1       0
question2       0
is_duplicate    0
dtype: int64
After dropna shape: (49847, 3)

Label counts:
 is_duplicate
0.0    31254
1.0    18593
Name: count, dtype: int64

Label proportions:
 is_duplicate
0.0    0.627
1.0    0.373
Name: count, dtype: float64

question1 token length summary:
count    49847.00
mean        10.93
std          5.44
min          1.00
50%         10.00
75%         13.00
90%         18.00
95%         22.00
99%         29.00
max        125.00
Name: question1, dtype: float64

question2 token length summary:
count    49847.00
mean        11.17
std          6.34
min          1.00
50%         10.00
75%         13.00
90%         19.00
95%         24.00
99%         33.00
max        237.00
Name: question2, dtype: float64

Exact string-match rate: 0.0


## Step 2 — Text preprocessing

We apply lightweight preprocessing to reduce noise and make the vocabulary more consistent:

- **Lowercasing**: treats “Apple” and “apple” as the same token.
- **Removing punctuation/special characters**: avoids creating many rare tokens.
- **Stopword removal** (NLTK English stopwords): removes very common words (e.g., “the”, “is”) that often add little signal.

**Output of this step**: two cleaned columns `q1_clean` and `q2_clean` that are used for feature extraction.

In [7]:
print("2. Preprocessing text... (This may take a minute or two)")

#Ensure NLTK stopwords are available (works even on fresh machines)
try:
    stop_words = set(stopwords.words("english"))
except LookupError:
    nltk.download("stopwords")
    stop_words = set(stopwords.words("english"))

def preprocess_text(text: str) -> str:
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    words = [w for w in text.split() if w not in stop_words]
    return " ".join(words)

#Apply preprocessing
df["q1_clean"] = df["question1"].apply(preprocess_text)
df["q2_clean"] = df["question2"].apply(preprocess_text)
print("-> Preprocessing complete!\n")

2. Preprocessing text... (This may take a minute or two)


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


-> Preprocessing complete!



### Notes on preprocessing choices

This baseline intentionally keeps preprocessing simple to stay fast and reproducible.

- **Numbers** are currently kept (because the regex keeps `0-9`). This can help for questions like “iPhone 12 vs iPhone 13”.
- **No stemming/lemmatization** is used here. We keep the surface form to avoid introducing extra dependencies/complexity.

If later experiments use embeddings/Transformers, we may reduce preprocessing because those models often perform best with minimal text cleaning.

## Step 3 — Feature extraction + Train/Test split

### Train/Test split
We split the dataset into **80% training** and **20% testing**, using **stratification** on `is_duplicate` so both splits keep a similar duplicate/non-duplicate ratio.

### TF‑IDF representation
We use **TF‑IDF** with up to **5000 features** and **(1,2)-grams**:
- Unigrams capture individual words.
- Bigrams help capture short phrases (e.g., “how to”, “best way”).

### Pair feature construction
Because the task is about *similarity between two questions*, we combine the two TF‑IDF vectors using:

- **Absolute difference**: \(|v_1 - v_2|\)

This is **order-invariant** (swapping the two questions gives the same features), which fits the duplicate-detection setting.


In [8]:
print("3. Splitting data and extracting TF-IDF features...")

#80/20 train-test split (stratified because the dataset is imbalanced)
X_train, X_test, y_train, y_test = train_test_split(
    df[["q1_clean", "q2_clean"]],
    df["is_duplicate"],
    test_size=0.2,
    random_state=42,
    stratify=df["is_duplicate"],
)

#Initialize TF-IDF Vectorizer (limit features for a fast baseline)
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))

#Fit only on training text to avoid leakage
all_train_questions = pd.concat([X_train["q1_clean"], X_train["q2_clean"]], axis=0)
tfidf.fit(all_train_questions)

#Transform questions into numerical vectors
q1_train_tfidf = tfidf.transform(X_train["q1_clean"])
q2_train_tfidf = tfidf.transform(X_train["q2_clean"])
q1_test_tfidf = tfidf.transform(X_test["q1_clean"])
q2_test_tfidf = tfidf.transform(X_test["q2_clean"])

#Pair feature: absolute difference (order-invariant)
X_train_features = abs(q1_train_tfidf - q2_train_tfidf)
X_test_features = abs(q1_test_tfidf - q2_test_tfidf)

print(f"-> Training features shape: {X_train_features.shape}")
print(f"-> Duplicate rate (train): {y_train.mean():.3f}")
print(f"-> Duplicate rate (test):  {y_test.mean():.3f}\n")

3. Splitting data and extracting TF-IDF features...
-> Training features shape: (39877, 5000)
-> Duplicate rate (train): 0.373
-> Duplicate rate (test):  0.373



## Step 4 — Model training (Logistic Regression)

We train a **Logistic Regression** classifier on the engineered TF‑IDF pair features.

**Why Logistic Regression?**
- Strong baseline for high-dimensional sparse text features.
- Fast to train on large datasets.
- Outputs calibrated-ish probabilities (useful for ROC‑AUC/PR‑AUC).

**Imbalance handling**
Duplicate datasets are typically imbalanced. We set `class_weight="balanced"` so the model does not over-favor the majority class.

In [9]:
print("4. Training Baseline Logistic Regression Model...")

#Helpful for imbalanced labels; keeps the model from defaulting to majority class
baseline_model = LogisticRegression(
    max_iter=2000,
    random_state=42,
    class_weight="balanced",
    n_jobs=None,
)

baseline_model.fit(X_train_features, y_train)
print("-> Training Complete!\n")

4. Training Baseline Logistic Regression Model...
-> Training Complete!



## Step 5 — Evaluation

I evaluate on the held-out **test set**.

**Metrics used**
- **Accuracy**: simple overall correctness, but can be misleading with imbalanced labels.
- **Majority baseline accuracy**: accuracy if we always predict the most common class (a minimum bar).
- **ROC‑AUC**: how well the model ranks duplicates above non-duplicates across all thresholds.
- **PR‑AUC (Average Precision)**: more informative than ROC‑AUC when the positive class is relatively rare.
- **Classification report + confusion matrix**: shows precision/recall/F1 and the types of errors.

In [10]:
print("5. Evaluating the model on unseen test data...")

#Predictions
y_pred = baseline_model.predict(X_test_features)
y_proba = baseline_model.predict_proba(X_test_features)[:, 1]

#A simple baseline for context: always predict the majority class (0)
majority_pred = np.zeros_like(y_test)

acc_model = accuracy_score(y_test, y_pred)
acc_majority = accuracy_score(y_test, majority_pred)
roc = roc_auc_score(y_test, y_proba)
ap = average_precision_score(y_test, y_proba)

results = pd.DataFrame(
    {
        "Metric": ["Accuracy", "ROC-AUC", "PR-AUC (Avg Precision)"],
        "Majority baseline (always 0)": [acc_majority, np.nan, np.nan],
        "LogReg + TF-IDF |diff|": [acc_model, roc, ap],
    }
)

print("\n" + "=" * 70)
print("RESULTS SUMMARY (Test Set)")
print("=" * 70)
print(results.to_string(index=False, float_format=lambda x: f"{x:.4f}" if np.isfinite(x) else ""))

#Detailed per-class metrics
print("\n" + "-" * 70)
print("CLASSIFICATION REPORT")
print("-" * 70)
print(classification_report(y_test, y_pred, target_names=["Not Duplicate", "Duplicate"]))

#Confusion matrix as a labeled table
cm = confusion_matrix(y_test, y_pred)
cm_df = pd.DataFrame(
    cm,
    index=pd.Index(["Actual: Not Duplicate", "Actual: Duplicate"], name=""),
    columns=["Pred: Not Duplicate", "Pred: Duplicate"],
)

print("\n" + "-" * 70)
print("CONFUSION MATRIX")
print("-" * 70)
print(cm_df.to_string())

5. Evaluating the model on unseen test data...

RESULTS SUMMARY (Test Set)
                Metric  Majority baseline (always 0)  LogReg + TF-IDF |diff|
              Accuracy                        0.6270                  0.7036
               ROC-AUC                           NaN                  0.7905
PR-AUC (Avg Precision)                           NaN                  0.6683

----------------------------------------------------------------------
CLASSIFICATION REPORT
----------------------------------------------------------------------
               precision    recall  f1-score   support

Not Duplicate       0.83      0.67      0.74      6251
    Duplicate       0.58      0.77      0.66      3719

     accuracy                           0.70      9970
    macro avg       0.70      0.72      0.70      9970
 weighted avg       0.73      0.70      0.71      9970


----------------------------------------------------------------------
CONFUSION MATRIX
------------------------------

## Interpretation (Baseline results)

- **Is the model better than the majority-class baseline?** If yes, it is learning signal beyond class imbalance.


### Main limitations
- TF‑IDF cannot capture deeper paraphrases well (same meaning, different words).
- No explicit modeling of word order beyond short n-grams.

### Next experiments (for NB2 / NB3)
- **NB2**: Sentence Transformer embeddings + a shallow classifier.
- **NB3**: Fine-tune a Transformer (BERT/DeBERTa) on question pairs.

These should improve paraphrase understanding compared to TF‑IDF.